In [14]:
import os
import numpy as np
import torch
import torch.nn as nn

In [3]:
class Dictionary:

    def __init__(self):

        self.word2indx = {}
        self.indx2word = {}
        self.indx = 0

    def add_word(self, word):

        if word not in self.word2indx:

            self.word2indx[word] = self.indx
            self.indx2word[self.indx] = word
            self.indx += 1

    def __len__(self):
        return len(self.word2indx)

In [4]:
class ProcessText:

    def __init__(self):
        self.dict = Dictionary()

    def get_data(self, path, batch_size = 20):
        with open(path, "r") as f:
            tokens = 0
            for line in f:
                words = line.split() + ["<eos>"]
                tokens += len(words)
                for word in words:
                    self.dict.add_word(word)

        rep_tensor = torch.zeros(size=(tokens,))
        index = 0
        with open(path, "r") as f:
            for line in f:
                words = line.split() + ["<eos>"]
                for word in words:
                    rep_tensor[index] = self.dict.word2indx[word]
                    index += 1
    
        num_batches = rep_tensor.shape[0] // batch_size
        rep_tensor = rep_tensor[:num_batches*batch_size]
        rep_tensor = rep_tensor.view(batch_size, -1)

        return rep_tensor

In [5]:
corpus = ProcessText()

In [19]:
vocab_size = len(corpus.dict)

In [6]:
data = corpus.get_data("alice.txt", 20)

In [7]:
embed_size = 128
hidden_size = 1024
num_layers = 1
num_epochs = 20
batch_size = 20
time_steps = 30
learning_rate = 0.002

In [8]:
print(data.shape)

torch.Size([20, 1484])


In [20]:
print(vocab_size)

5290


In [11]:
num_batches = data.shape[1] // time_steps

In [12]:
num_batches

49

In [45]:
class TextGenerator(nn.Module):

    def __init__(self, vocab_size, embedding_size, hidden_size, num_layers):

        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(input_size=embed_size, hidden_size=hidden_size, num_layers=num_layers,batch_first=True)
        self.linear = nn.Linear(in_features=hidden_size, out_features=vocab_size)

    def forward(self, x, h):

        x = self.embed(x)
        #input shape of the tensor needs to (batch_size, seq_len or timesteps, embedding_size)
        out, (h, c) = self.lstm(x, h)
        #Reshape the output from (samples, timesteps, output_features) to an appropriate shape for the FC Layer (batch, features)
        #(batch_size*timesteps, embed_size)
        out = out.reshape(out.shape[0]*out.shape[1], out.shape[2])
        out = self.linear(out)

        return out, (h, c)

In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [47]:
model = TextGenerator(vocab_size=vocab_size, embedding_size=embed_size, hidden_size=hidden_size, num_layers=num_layers).to(device)

In [48]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [49]:
data = data.long().to(device)

In [51]:
#TRAINING LOOP

for epoch in range(num_epochs):

    states = (torch.zeros(num_layers,batch_size,hidden_size,device=device),
              torch.zeros(num_layers,batch_size, hidden_size, device=device))

    for i in range(0, data.shape[1] - time_steps, time_steps):

        inputs = data[:, i:i+time_steps]
        targets = data[:, i+1:i+1+time_steps]
        
        outputs, _ = model(inputs,states)
        loss = criterion(outputs, targets.reshape(-1))

        model.zero_grad()
        loss.backward()

        nn.utils.clip_grad_norm(model.parameters(), 0.5)

        optimizer.step()

        steps = (i+1 // time_steps)

        if steps % 100 == 0:

            print(f"{epoch+1}/{num_epochs}: Loss:{loss.item():.4f}")

C:\Users\rajme\AppData\Local\Temp\ipykernel_92380\3798400740.py:19: FutureWarning: `torch.nn.utils.clip_grad_norm` is now deprecated in favor of `torch.nn.utils.clip_grad_norm_`.
  nn.utils.clip_grad_norm(model.parameters(), 0.5)


1/20: Loss:0.0943
1/20: Loss:0.1147
1/20: Loss:0.0947
1/20: Loss:0.0938
1/20: Loss:0.0859
2/20: Loss:0.0763
2/20: Loss:0.1028
2/20: Loss:0.0863
2/20: Loss:0.0858
2/20: Loss:0.0903
3/20: Loss:0.0721
3/20: Loss:0.0967
3/20: Loss:0.0820
3/20: Loss:0.0831
3/20: Loss:0.0732
4/20: Loss:0.0689
4/20: Loss:0.0964
4/20: Loss:0.0798
4/20: Loss:0.0812
4/20: Loss:0.0777
5/20: Loss:0.0670
5/20: Loss:0.0928
5/20: Loss:0.0785
5/20: Loss:0.0798
5/20: Loss:0.0689
6/20: Loss:0.0657
6/20: Loss:0.0933
6/20: Loss:0.0774
6/20: Loss:0.0787
6/20: Loss:0.0749
7/20: Loss:0.0646
7/20: Loss:0.0908
7/20: Loss:0.0766
7/20: Loss:0.0778
7/20: Loss:0.0674
8/20: Loss:0.0637
8/20: Loss:0.0911
8/20: Loss:0.0758
8/20: Loss:0.0770
8/20: Loss:0.0728
9/20: Loss:0.0630
9/20: Loss:0.0894
9/20: Loss:0.0753
9/20: Loss:0.0763
9/20: Loss:0.0666
10/20: Loss:0.0624
10/20: Loss:0.0895
10/20: Loss:0.0747
10/20: Loss:0.0757
10/20: Loss:0.0713
11/20: Loss:0.0619
11/20: Loss:0.0885
11/20: Loss:0.0743
11/20: Loss:0.0751
11/20: Loss:0.0663


In [37]:
inputs.shape

torch.Size([20, 30])

In [38]:
targets.shape

torch.Size([20, 30])

In [68]:
with torch.no_grad():

    with open("results.txt", "w") as f:
    
        inp = torch.randint(0, vocab_size, (1,1)).long().to(device)
        state = (torch.zeros(num_layers,1,hidden_size,device=device),
                      torch.zeros(num_layers,1, hidden_size, device=device))

        for i in range(500):
            
            output, _ = model(inp, state)
            prob = output.exp()
            word_id = torch.multinomial(prob, num_samples = 1).item()
            inp.fill_(word_id)

            word = corpus.dict.indx2word[word_id]
            word = "\n" if word == "<eos>" else word + " "
            f.write(word)

In [65]:
output.exp()

tensor([[0.0003, 0.0016, 0.0017,  ..., 0.0024, 0.0023, 0.0047]],
       device='cuda:0')